# 配置与导入

In [1]:
import copy
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding
from kernel.palu_attention import apply_rotary_pos_emb

# 超参
MODEL_PATH = "Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-whiten"
SEQ_LEN = 128
BATCH_SIZE = 8
NUM_STEPS = 2000
EVAL_EVERY = 200
MAX_TEST_WINDOWS = 10  # 每次快速 PPL 评估用多少个窗口
DATASET_NAME = "wikitext-2-raw-v1"  # wikitext-2-raw-v1

# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True

# 评估函数

In [22]:
### PPL 评估
def evaluate_ppl(model, tokenizer, dataset_name="wikitext2", split="test",
                 seqlen=2048, device="cuda", windows=None, nsamples=None):
    import torch
    import torch.nn as nn
    from datasets import load_dataset
    from tqdm import tqdm

    # 与 run_ppl_eval.py 一致的数据来源与切窗方式
    testdata = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="test",
    )
    testenc = tokenizer("\n\n".join(testdata["text"]), return_tensors="pt").input_ids

    # 与 run_ppl_eval.py 一致的 forward 与 loss 计算
    model = model.to(device)
    if isinstance(device, str):
        device = torch.device(device)

    nsamples = testenc.numel() // seqlen if nsamples is None else nsamples
    use_cache = model.config.use_cache
    model.config.use_cache = False
    model.eval()

    nlls = []
    with torch.no_grad():
        for i in tqdm(range(nsamples)):
            batch = testenc[:, (i * seqlen):((i + 1) * seqlen)].to(device)
            outputs = model(batch)
            logits = outputs.logits
            shift_logits = logits[:, :-1, :]
            shift_labels = testenc[:, (i * seqlen):((i + 1) * seqlen)][:, 1:].to(device)
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1)
            )
            neg_log_likelihood = loss.float() * seqlen
            nlls.append(neg_log_likelihood)

    ppl = torch.exp(torch.stack(nlls).sum() / (len(nlls) * seqlen)).item()
    model.config.use_cache = use_cache
    # example_generation(model, tokenizer, device)
    return ppl

def example_generation(model, tokenizer, device):
    # Example generation (no KV cache to avoid shape mismatch)
    prompt = "Why research is so hard?"
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        prev_use_cache = getattr(model.config, "use_cache", None)
        model.config.use_cache = False  # disable cache globally
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=False,            # disable cache in generate
        )
        if prev_use_cache is not None:
            model.config.use_cache = prev_use_cache

    gen_text = tokenizer.decode(gen_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print("=== Example Prompt ===")
    print(prompt)
    print(gen_text)
    return


### Zero-shot OpenBookQA 准确率评估
def zero_shot_eval(model, tokenizer, tasks, *,
                                      batch_size: int = 8,
                                      max_length: int = 4096,
                                      limit: int | None = None,
                                      return_full: bool = False):
    """
    Same core logic as run_lm_eval.py but uses an already-loaded model/tokenizer.
    - Wraps model/tokenizer with HFLM
    - Runs lm_eval.simple_evaluate on the given tasks
    - Prints the results table and returns results['results'] by default
    - res = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
    """
    import torch
    import lm_eval
    from lm_eval.models.huggingface import HFLM
    from lm_eval.tasks import TaskManager
    from lm_eval.utils import make_table

    # normalize tasks
    task_list = [t.strip() for t in tasks.split(",")] if isinstance(tasks, str) else list(tasks)

    model.seqlen = max_length
    lm_obj = HFLM(pretrained=model, tokenizer=tokenizer, add_bos_token=False, batch_size=batch_size)
    task_manager = TaskManager()

    with torch.no_grad():
        results = lm_eval.simple_evaluate(
            model=lm_obj,
            tasks=task_list,
            task_manager=task_manager,
            log_samples=False,
            limit=limit,
        )

    print(make_table(results))
    return results if return_full else results["results"]



## 1) 加载model和dataset

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("模型已加载。")

layer0 = model.model.layers[0].self_attn
layer0_original  = copy.deepcopy(layer0)
print("layer0_original 已存档")

# 预缓存测试集以加速 PPL
ds_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
ds_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
test_texts = [ex["text"] for ex in ds_test if ex["text"].strip()]
test_text_cat = "\n\n".join(test_texts)
test_tok = tokenizer(test_text_cat, return_tensors="pt")
test_ids_all = test_tok.input_ids[0]
print("数据集已缓存。")

# 重置模型
def reset_model(model, layer0_original):
    model.model.layers[0].self_attn = copy.deepcopy(layer0_original)
    layer0 = model.model.layers[0].self_attn
    print("Model reset to original")
    return layer0

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

模型已加载。
layer0_original 已存档
数据集已缓存。


## 2) 训练辅助函数


In [12]:
def sample_batch(tokenizer, batch_size=BATCH_SIZE, seq_len=SEQ_LEN, device=device):
    texts = []
    while len(texts) < batch_size:
        t = ds_train[np.random.randint(len(ds_train))]["text"].strip()
        if t:
            texts.append(t)
    tok = tokenizer(
        texts, max_length=seq_len, truncation=True, padding="max_length", return_tensors="pt"
    )
    return tok.input_ids.to(device)

@torch.no_grad()
def get_hidden_normed(input_ids):
    # 仅第0层的前处理：embed -> input_layernorm
    embed = model.model.embed_tokens
    ln0 = model.model.layers[0].input_layernorm
    hs = embed(input_ids)
    hs = ln0(hs)
    return hs  # [B, T, H]

def alignment_loss(input_ids):
    B, T = input_ids.shape
    pos_ids = torch.arange(T, device=device).unsqueeze(0).expand(B, -1)

    # 预处理 hidden_states
    hs = get_hidden_normed(input_ids)  # [B, T, H]

    # 计算 cos/sin（按 K 的 head_dim）
    # 注意：rotary_full 的前向需要一个“形状提示”，这里用 K 的最终形状来生成 cos/sin
    # 用一个 dummy tensor 只为生成 cos/sin；下方实际应用在不同张量上
    # Rotary Embedding（放在第0层所在设备）
    rotary_full = LlamaRotaryEmbedding(config=layer0.config).to(device)
    head_dim = layer0.head_dim
    num_kv = layer0.num_key_value_heads
    dummy = torch.empty(B, num_kv, T, head_dim, device=device, dtype=hs.dtype)
    cos, sin = rotary_full(dummy, pos_ids)  # 形状 [B, T, head_dim]

    # === 目标：PALU 路径（RoPE(x@U@V)) ===
    with torch.no_grad():
        k_lat_palu = layer0_original.k_proj.project_to_latent(hs)  # [B, T, total_latent_k]
        k_palu = layer0_original.k_proj.reconstruct(k_lat_palu)    # [B, T, num_kv*head_dim]
        k_palu = k_palu.view(B, T, num_kv, head_dim).transpose(1, 2)  # [B, heads, T, head_dim]
        _, k_palu_rope = apply_rotary_pos_emb(None, k_palu, cos, sin)  # 对 key 施加 RoPE

    # === 预测：HACK 路径（RoPE(x@U)@V) ===
    k_lat_hack = layer0.k_proj.project_to_latent(hs.float())  # [B, T, total_latent_k]
    latent_dim = k_lat_hack.shape[-1] // num_kv
    k_lat_hack = k_lat_hack.view(B, T, num_kv, latent_dim).transpose(1, 2)  # [B, heads, T, latent_dim]
    # 在 latent 维度上截断 cos/sin 后应用 RoPE
    _, k_lat_hack_rope = apply_rotary_pos_emb(None, k_lat_hack, cos[..., :latent_dim], sin[..., :latent_dim])
    # 重构回 key states
    k_lat_hack_rope = k_lat_hack_rope.transpose(1, 2).reshape(B, T, -1)  # [B, T, total_latent_k]
    k_hack = layer0.k_proj.reconstruct(k_lat_hack_rope).view(B, T, num_kv, head_dim).transpose(1, 2)  # [B, heads, T, head_dim]

    # MSE 对齐
    return nn.functional.mse_loss(k_hack, k_palu_rope.float())

## 3) 训练循环：优化 U/V（仅 K），并周期性评估整模 PPL

In [23]:
# 初始 PPL
layer0.to(dtype=torch.float16)
base_ppl = evaluate_ppl(model, tokenizer, DATASET_NAME, "test", 2048, device)
print(f"Baseline (rope = {layer0.rope_latent}) PPL: {base_ppl:.4f}")
layer0.rope_latent = True  # Set HACK AttentionRoPE(x@U)@V
base_ppl_hack = evaluate_ppl(model, tokenizer, DATASET_NAME, "test", 2048, device)
print(f"HACK Attention (rope = {layer0.rope_latent}) PPL: {base_ppl_hack:.4f}")


# 训练
train_params = [layer0.k_proj.VT.weight, layer0.k_proj.U[0].weight, layer0.k_proj.U[1].weight]
for n, p in layer0.named_parameters(): p.requires_grad_(False)
for p in train_params: p.requires_grad_(True) # 只训练 hack0 的 k_proj 中的 U 和 VT
optimizer = torch.optim.AdamW(train_params, lr=5e-4, weight_decay=1e-6, eps=1e-8)
print("构建训练目标 train_params")
#训练循环
loss_hist = []
ppl_hist = []
best_ppl = float('inf')
best_layer0 = None
for step in tqdm(range(1, NUM_STEPS + 1), desc="Aligning K (RoPE latent vs full)"):
    input_ids = sample_batch(tokenizer, BATCH_SIZE, SEQ_LEN, device)
    optimizer.zero_grad(set_to_none=True)
    layer0.to(dtype=torch.float32)
    loss = alignment_loss(input_ids)
    if torch.isfinite(loss):
        loss.backward()
        torch.nn.utils.clip_grad_norm_(train_params, 0.05)
        optimizer.step()
        loss_hist.append(float(loss.item()))
    layer0.to(dtype=torch.float16)
    if step % EVAL_EVERY == 0 or step <= 10:
        ppl = evaluate_ppl(model, tokenizer, seqlen=2048, windows=MAX_TEST_WINDOWS, nsamples=10)
        ppl_hist.append(ppl)
        print(f"Step {step}: align_loss={loss.item():.6e}, evaluate_ppl with 10 samples={ppl:.4f}")
        if ppl < best_ppl:
            best_ppl = ppl
            best_layer0 = copy.deepcopy(model.model.layers[0].self_attn)

# 结束后做一次完整 PPL
model.model.layers[0].self_attn = best_layer0
final_ppl = evaluate_ppl(model, tokenizer, DATASET_NAME, "test", 2048, device)
print(f"Final (HACK@layer0) PPL: {final_ppl:.4f}")

100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:27<00:00,  5.07it/s]


Baseline (rope = False) PPL: 8.7749


100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:27<00:00,  5.11it/s]


HACK Attention (rope = True) PPL: 4035.8945
构建训练目标 train_params


Aligning K (RoPE latent vs full):   0%|                                               | 1/2000 [00:07<3:54:55,  7.05s/it]

...

Step 1800: align_loss=2.542635e-01, evaluate_ppl with 10 samples=794.2067


Aligning K (RoPE latent vs full): 100%|██████████████████████████████████████████████| 2000/2000 [02:17<00:00, 14.51it/s]


Step 2000: align_loss=2.324854e-01, evaluate_ppl with 10 samples=728.2361


100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:27<00:00,  5.07it/s]


Final (HACK@layer0) PPL: 24.2197


## Optional) - 评估 Initial Baseline - PPL & OpenBookQA


In [ ]:
###===============ppl评估===============###
# 评估 Original ppl
layer0 = reset_model(model, layer0_original)
ppl_original = evaluate_ppl(model, tokenizer, seqlen=2048, windows=10, device=device)
print(f"💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻Evaluating LLAMA with original model: PPL= {ppl_original:.4f}")
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")
# 评估HACK ppl before finetune
layer0.rope_latent = True
ppl_hack = evaluate_ppl(model, tokenizer, seqlen=2048, windows=10, device=device)
print(f"💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻Evaluating LLAMA with HACK model: PPL= {ppl_hack:.4f}")
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")
# 评估HACK ppl after finetune
model.model.layers[0].self_attn = best_layer0
final_ppl = evaluate_ppl(model, tokenizer, DATASET_NAME, "test", 2048, device)
print(f"Final (HACK@layer0) PPL: {final_ppl:.4f}")
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")


###===============zero-shot OpenBookQA===============###
# 评估original accuracy
layer0 = reset_model(model, layer0_original)
res_original = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻Evaluating LLAMA with original model: accuracy= {res_original['openbookqa']['acc']:.4f}")
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")
# 评估HACK accuracy
layer0.rope_latent = True
res_hack = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻Evaluating LLAMA with HACK model: accuracy= {res_hack['openbookqa']['acc']:.4f}")
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")
# 评估HACK accuracy after finetune
model.model.layers[0].self_attn = best_layer0
res_hack_finetune = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])
print(f"💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻Evaluating LLAMA with HACK model after finetune: accuracy= {res_hack_finetune['openbookqa']['acc']:.4f}")
print("\n--------------------------------------------------------------------------------------------------------------------------------\n")